# Natural Questions (N=100) Logprob Capture and Verification

This notebook runs Natural Questions inference with activation logging enabled, then verifies response logprobs in two ways:
1. Directly from the Zarr store arrays
2. Through `ActivationParser` + `ActivationDataset`

In [ ]:
from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
import zarr

from scripts.run_with_server import run_experiment
from utils import lm
from activation_logging.activations_logger import ActivationsLogger
from activation_logging.activation_parser import ActivationParser

model = "meta-llama/Llama-3.1-8B-Instruct"
task = "naturalquestions"
step = "inference"
N = 100

inference_method = "vllm"
max_tokens = 64
temperature = 0.0

logger_type = "zarr"
run_root = Path("output/natural_questions_logprob_n100")
activations_path = run_root / "activations.zarr"
log_file = run_root / "server.log"

data_dir = "external/LLMsKnow/data"
output_dir = str(run_root)
generations_file_path = run_root / "natural_questions" / model.split("/")[-1] / "generation.jsonl"
eval_results_path = run_root / "natural_questions" / model.split("/")[-1] / "eval_results.json"

host = "0.0.0.0"
port = 8000

for p in [run_root, activations_path.parent, generations_file_path.parent]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["ACTIVATION_LOGPROBS_ENABLED"] = "1"
os.environ["ACTIVATION_LOGPROBS_TOPK"] = "20"

print(f"Run root: {run_root.resolve()}")
print(f"Activations path: {activations_path}")
print(f"Generations path: {generations_file_path}")

In [ ]:
# Use a clean server instance so logprob env settings are definitely applied.
force_clean_server_start = True
if force_clean_server_start:
    lm.kill_process_on_port(port)

run_experiment(
    step=step,
    task=task,
    model=model,
    inference_method=inference_method,
    N=N,
    max_tokens=max_tokens,
    temperature=temperature,
    logger_type=logger_type,
    activations_path=str(activations_path),
    log_file=str(log_file),
    data_dir=data_dir,
    output_dir=output_dir,
    generations_file_path=str(generations_file_path),
    eval_results_path=str(eval_results_path),
    host=host,
    port=port,
)

print("Inference run completed.")
print(f"Generated file exists: {generations_file_path.exists()}")
print(f"Zarr path exists: {activations_path.exists()}")

In [ ]:
# Direct Zarr verification: logprob arrays exist and valid token positions are finite.
root = zarr.open_group(str(activations_path), mode="r")
arrays = root["arrays"]

required = [
    "response_len",
    "response_token_ids",
    "response_token_logprobs",
    "response_topk_token_ids",
    "response_topk_logprobs",
]
missing = [name for name in required if name not in arrays]
assert not missing, f"Missing required arrays in Zarr: {missing}"

response_len = arrays["response_len"][:].astype(np.int32)
token_ids = arrays["response_token_ids"][:]
token_lp = arrays["response_token_logprobs"][:]
topk_lp = arrays["response_topk_logprobs"][:]

num_samples = int(response_len.shape[0])
num_samples_with_response = int((response_len > 0).sum())

max_tokens_axis = token_lp.shape[1]
valid_mask = np.arange(max_tokens_axis)[None, :] < response_len[:, None]
valid_token_ids = token_ids[valid_mask]
valid_token_lp = token_lp[valid_mask]
valid_topk_lp = topk_lp[valid_mask]

token_non_finite = int((~np.isfinite(valid_token_lp)).sum())
topk_non_finite = int((~np.isfinite(valid_topk_lp)).sum())
token_ids_negative = int((valid_token_ids < 0).sum())

summary = {
    "num_samples": num_samples,
    "num_samples_with_response": num_samples_with_response,
    "valid_token_logprobs_count": int(valid_token_lp.size),
    "valid_topk_logprobs_count": int(valid_topk_lp.size),
    "token_non_finite_count": token_non_finite,
    "topk_non_finite_count": topk_non_finite,
    "negative_token_ids_in_valid_positions": token_ids_negative,
}
print(json.dumps(summary, indent=2))

assert num_samples > 0, "No samples found in Zarr store."
assert num_samples_with_response > 0, "No samples with non-zero response length."
assert valid_token_lp.size > 0, "No valid token logprobs to validate."
assert token_non_finite == 0, f"Found non-finite token logprobs: {token_non_finite}"
assert topk_non_finite == 0, f"Found non-finite top-k logprobs: {topk_non_finite}"
assert token_ids_negative == 0, f"Found negative token IDs in valid positions: {token_ids_negative}"

print("Direct Zarr verification passed: logprobs exist and are finite for valid positions.")

In [ ]:
# ActivationParser + Dataset verification for response logprobs.
assert generations_file_path.exists(), f"Missing generations file: {generations_file_path}"

records = []
with generations_file_path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

gendf = pd.DataFrame(records)
assert len(gendf) > 0, "Generations file is empty."
assert "prompt" in gendf.columns, "Generations JSONL does not contain 'prompt'."

logger_ro = ActivationsLogger(lmdb_path=str(activations_path), read_only=True, verbose=False)
keys = logger_ro.list_entries()
logger_ro.close()
base_keys = {k.split("_")[0] for k in keys}

gendf["prompt_hash"] = gendf["prompt"].apply(
    lambda p: hashlib.sha256(str(p).encode("utf-8")).hexdigest()
)
df = gendf[gendf["prompt_hash"].isin(base_keys)].copy()
assert len(df) > 0, "No overlap between generations and Zarr entry keys."

df["halu"] = False
df["split"] = "train"
if len(df) >= 5:
    df.loc[df.index[::5], "split"] = "test"

placeholder_dir = run_root / "parser_placeholders"
placeholder_dir.mkdir(parents=True, exist_ok=True)
inference_placeholder = placeholder_dir / "inference_placeholder.jsonl"
eval_placeholder = placeholder_dir / "eval_placeholder.json"
if not inference_placeholder.exists():
    inference_placeholder.write_text("{}\n", encoding="utf-8")
if not eval_placeholder.exists():
    eval_placeholder.write_text("{}\n", encoding="utf-8")

parser = ActivationParser(
    inference_json=str(inference_placeholder),
    eval_json=str(eval_placeholder),
    activations_path=str(activations_path),
    df=df,
    logger_type="zarr",
    verbose=False,
)

dataset = parser.get_dataset(
    split="train",
    relevant_layers=list(range(16, 30)),
    pad_length=64,
    min_target_layers=2,
    num_views=2,
    include_response_logprobs=True,
    response_logprobs_top_k=20,
)

assert len(dataset) > 0, "ActivationDataset(train) is empty."

checked_samples = 0
tokens_checked = 0
topk_values_checked = 0
max_samples_to_check = min(20, len(dataset))

for i in range(max_samples_to_check):
    sample = dataset[i]
    mask = sample["response_logprob_mask"].cpu().numpy().astype(bool)
    token_lp_ds = sample["response_token_logprobs"].cpu().numpy()
    topk_lp_ds = sample["response_topk_logprobs"].cpu().numpy()

    valid_token_lp_ds = token_lp_ds[mask]
    valid_topk_lp_ds = topk_lp_ds[mask, :]

    if valid_token_lp_ds.size == 0:
        continue

    assert np.isfinite(valid_token_lp_ds).all(), f"Non-finite token logprobs in dataset sample {i}"
    assert np.isfinite(valid_topk_lp_ds).all(), f"Non-finite top-k logprobs in dataset sample {i}"

    checked_samples += 1
    tokens_checked += int(valid_token_lp_ds.size)
    topk_values_checked += int(valid_topk_lp_ds.size)

assert checked_samples > 0, "No dataset samples had valid response logprobs."

parser_summary = {
    "dataset_length": int(len(dataset)),
    "checked_samples": int(checked_samples),
    "tokens_checked": int(tokens_checked),
    "topk_values_checked": int(topk_values_checked),
}
print(json.dumps(parser_summary, indent=2))
print("ActivationParser/Dataset verification passed: logprobs are present and finite.")